In [1]:
%matplotlib qt
import sys
import glob
import os
from pathlib import Path

src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import torch
import json
import gymnasium as gym
from envs.env import MiniGridEnvWrapper

from models.ppo_lstm import PPOLSTMAgent
from models.ride import RIDEAgent
from training.train import train_ppo_lstm, train_ride
from training.evaluate import evaluate_agent

/Users/qi/Desktop/CS238 Project/minigrid-with-curiosity/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [6]:
env_ids = [
    "MiniGrid-DoorKey-9x9-v0",
    "MiniGrid-SimpleCrossingS9N1-v0",
    "MiniGrid-LavaCrossingS9N2-v0",
    "MiniGrid-MultiRoom-N2-S4-v0",
    "MiniGrid-MultiRoom-N4-S5-v0",
    "MiniGrid-KeyCorridorS3R3-v0"
]
env = MiniGridEnvWrapper(env_ids[4], render_mode='rgb_array')

obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

num_steps = 100
for step in range(num_steps):
    action = env.action_space.sample()
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {step + 1}: Action={action}, Reward={reward}, Done={terminated or truncated}")
    
    env.display_interactive()
    
    if terminated or truncated:
        print(f"Episode finished after {step + 1} steps!")
        obs, info = env.reset()
        print("Environment reset for next episode")
        env.display_interactive()
    time.sleep(0.01)
env.close()

Initial observation shape: (7, 7, 3)
Action space: Discrete(7)
Step 1: Action=1, Reward=0, Done=False
Step 2: Action=5, Reward=0, Done=False
Step 3: Action=2, Reward=0, Done=False
Step 4: Action=6, Reward=0, Done=False
Step 5: Action=5, Reward=0, Done=False
Step 6: Action=1, Reward=0, Done=False
Step 7: Action=1, Reward=0, Done=False
Step 8: Action=6, Reward=0, Done=False
Step 9: Action=0, Reward=0, Done=False
Step 10: Action=3, Reward=0, Done=False
Step 11: Action=1, Reward=0, Done=False
Step 12: Action=4, Reward=0, Done=False
Step 13: Action=0, Reward=0, Done=False
Step 14: Action=4, Reward=0, Done=False
Step 15: Action=2, Reward=0, Done=False
Step 16: Action=5, Reward=0, Done=False
Step 17: Action=2, Reward=0, Done=False
Step 18: Action=4, Reward=0, Done=False
Step 19: Action=3, Reward=0, Done=False
Step 20: Action=4, Reward=0, Done=False
Step 21: Action=2, Reward=0, Done=False
Step 22: Action=4, Reward=0, Done=False
Step 23: Action=4, Reward=0, Done=False
Step 24: Action=3, Reward=

In [3]:
agent = train_ppo_lstm(
        env=env,
        experiment_name="ppo_lstm_simple_crossing",
        num_iterations=300,
        print_interval=1,  # Print every 10 iterations for cleaner output
        steps_per_iteration=2048,
        save_interval=100,
        device="cpu",  # or "cuda" or "mps"
        lr=3e-4,
        gamma=0.99,
        ppo_epochs=4,
        ppo_minibatch_size=4,
        hidden_size=256,
    )

Logging to: ../runs/ppo_lstm_simple_crossing_20251116_191038

Starting training: ppo_lstm_simple_crossing_20251116_191038

[   1/300] Frames:   2,048 | Reward:    0.00 | Length:  324.0 | Loss: 0.0067/0.0000
[   2/300] Frames:   4,096 | Reward:    0.00 | Length:  324.0 | Loss: -0.0054/0.0000
[   3/300] Frames:   6,144 | Reward:    0.00 | Length:  324.0 | Loss: 0.0027/0.0000
[   4/300] Frames:   8,192 | Reward:    0.00 | Length:  324.0 | Loss: 0.0018/0.0000
[   5/300] Frames:  10,240 | Reward:    0.00 | Length:  324.0 | Loss: 0.0008/0.0000
[   6/300] Frames:  12,288 | Reward:    0.00 | Length:  324.0 | Loss: -0.0167/0.0000
[   7/300] Frames:  14,336 | Reward:    0.00 | Length:  324.0 | Loss: -0.0012/0.0000
[   8/300] Frames:  16,384 | Reward:    0.00 | Length:  324.0 | Loss: -0.0211/0.0000
[   9/300] Frames:  18,432 | Reward:    0.12 | Length:  288.0 | Loss: -0.0013/0.0021
[  10/300] Frames:  20,480 | Reward:    0.00 | Length:  324.0 | Loss: -0.0037/0.0000
[  11/300] Frames:  22,528 | Re

In [ ]:
from models import ppo_lstm

agent_class = PPOLSTMAgent
env = MiniGridEnvWrapper(env_ids[3], render_mode='rgb_array')
ppo_lstm_runs = sorted(glob.glob("../runs/ppo_lstm_multiroom_n2_*"), key=os.path.getmtime, reverse=True)
if ppo_lstm_runs:
    latest_run = ppo_lstm_runs[0]
    config_path = os.path.join(latest_run, "config.json")
    checkpoint_path = os.path.join("../checkpoints", os.path.basename(latest_run), "final_model.pt")
    
    print(f"Using checkpoint: {checkpoint_path}")
    print(f"Using config: {config_path}")
    
    results = evaluate_agent(agent_class, env, checkpoint_path, config_path)


In [7]:
# Train RIDE agent
ride_agent = train_ride(
    env = env,
    experiment_name="ride_multiroom_n4",
    num_iterations=200,
    print_interval=1,
    steps_per_iteration=2048,
    save_interval=50,
    device="cpu",  # or "cuda" or "mps"
    lr=3e-4,  
    gamma=0.99,
    ppo_epochs=4,
    ppo_minibatch_size=8,
    hidden_size=256, 
    entropy_coef=0.001,  
    intrinsic_reward_coef=0.01, 
    max_grad_norm=0.5,
    use_intrinsic_normalization=True, 
)


Logging to: ../runs/ride_multiroom_n4_20251116_194607

Starting RIDE training: ride_multiroom_n4_20251116_194607
Intrinsic reward coefficient: 0.01

[   1/200] Frames:   2,048 | Reward:    0.00 | Intrinsic:  0.488 | Length:  120.0 | Loss: 0.0302/0.0000 | FD: 0.0467 ID: 1.9458
  Actions: turn: 12.6%  turn: 14.3%  move: 14.6%  pick: 14.5%  drop: 15.9%  togg: 13.8%★ done: 14.4%  
  ✓ Toggle (door open) usage: 13.8% - OK
[   2/200] Frames:   4,096 | Reward:    0.00 | Intrinsic:  0.492 | Length:  120.0 | Loss: -0.0088/0.0000 | FD: 0.0329 ID: 1.9456
  Actions: turn: 20.1%  turn: 20.6%  move: 15.3%  pick: 11.8%  drop: 10.5%  togg: 11.0%★ done: 10.6%  
  ✓ Toggle (door open) usage: 11.0% - OK
[   3/200] Frames:   6,144 | Reward:    0.00 | Intrinsic:  0.421 | Length:  120.0 | Loss: 0.0010/0.0000 | FD: 0.0260 ID: 1.9431
  Actions: turn: 21.3%  turn: 21.7%  move: 14.8%  pick: 11.2%  drop: 11.1%  togg:  9.5%★ done: 10.4%  
  ✓ Toggle (door open) usage: 9.5% - OK
[   4/200] Frames:   8,192 | Reward

In [8]:
# Evaluate RIDE agent
ride_agent_class = RIDEAgent
env_ride_eval = MiniGridEnvWrapper(env_ids[1], render_mode='rgb_array')

ride_runs = sorted(glob.glob("../runs/ride_simple_crossing*"), key=os.path.getmtime, reverse=True)
if ride_runs:
    latest_run = ride_runs[0]
    config_path = os.path.join(latest_run, "config.json")
    checkpoint_path = os.path.join("../checkpoints", os.path.basename(latest_run), "final_model.pt")
    
    print(f"Using checkpoint: {checkpoint_path}")
    print(f"Using config: {config_path}")
    
    results = evaluate_agent(ride_agent_class, env_ride_eval, checkpoint_path, config_path)
else:
    print("No RIDE training runs found. Please train the agent first.")


Using checkpoint: ../checkpoints/ride_simple_crossing_20251115_201202/final_model.pt
Using config: ../runs/ride_simple_crossing_20251115_201202/config.json
Loading checkpoint from ../checkpoints/ride_simple_crossing_20251115_201202/final_model.pt...
Checkpoint loaded successfully!

Running 10 evaluation episodes...
Episode  1/10 | Steps: 100 (max) | Reward:   0.00
Episode  2/10 | Steps:  18 | Reward:   0.95
Episode  3/10 | Steps:  25 | Reward:   0.93
Episode  4/10 | Steps:  98 | Reward:   0.73
Episode  5/10 | Steps:  25 | Reward:   0.93
Episode  6/10 | Steps:  40 | Reward:   0.89
Episode  7/10 | Steps:  13 | Reward:   0.96
Episode  8/10 | Steps:  22 | Reward:   0.94
Episode  9/10 | Steps:  25 | Reward:   0.93
Episode 10/10 | Steps:  22 | Reward:   0.94

Evaluation Summary:
  Episodes:     10
  Mean Reward:  0.82 ± 0.28
  Min/Max:      0.00 / 0.96
  Mean Length:  38.80 ± 30.80
  Success Rate: 90.0%
